In [114]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

In [115]:
import re
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

from models.lstm_model import LSTM_Model
from utils import get_device,tokenize,encode_text,NLP_data_cleaning


# Parameters

In [116]:
checkpoint_path = "../../../checkpoints/sentiment_checkpoint.pt"

## Extract saved Objects and config

In [117]:
device = get_device()

checkpoint = torch.load(checkpoint_path, map_location=device)

In [118]:
model_config = checkpoint["model_config"]
vocab = checkpoint["vocab"]
label_map = checkpoint["label_map"]

In [119]:
preprocess_config = checkpoint.get("preprocess_config", {})
max_len = int(preprocess_config.get("seq_length", checkpoint.get("max_len", 40)))
pad_idx = int(preprocess_config.get("padding_idx", 0))
unk_idx = int(preprocess_config.get("unk_idx", 1))

In [120]:
id_to_label = {int(v): k for k, v in label_map.items()}

## Rebuild model

In [121]:
model = LSTM_Model(**model_config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
_= model.eval()

# Functions

## Function to preprocess data

In [122]:
def data_preprocess( texts ):

    texts = [ NLP_data_cleaning(text) for text in texts]
    
    X = [
        encode_text(text=t, vocab=vocab, unk_idx=unk_idx, pad_idx=pad_idx, max_len=max_len)
        for t in texts
    ]

    X_tensor = torch.tensor(X, dtype=torch.long, device=device)

    return X_tensor
    

## Function to predict

In [123]:
def predict_sentiment( X_tensor ):

    logits = model(X_tensor)
    probs = F.softmax(logits, dim=1)  
    
    pred_ids = probs.argmax(dim=1).cpu().tolist()
    pred_confs = probs.max(dim=1).values.cpu().tolist()

    results = []
    
    for text, cls_id, conf, p in zip(sample_texts, pred_ids, pred_confs, probs.cpu().tolist()):
        label = id_to_label.get(cls_id, str(cls_id))
        results.append(
            {
                "text": text,
                "pred_id": cls_id,
                "pred_label": label,
                "confidence": float(conf),
                "probs": [float(x) for x in p],
            }
        )    


    return results

## Main function to process and predict sentiment of the input list of sentense

In [124]:
def main_sentiment_predict(texts, actual = None):

    
    correct = 0
    incorrect = 0

    processed_X = data_preprocess( texts )
    
    result = predict_sentiment( processed_X )

    
    for item in result:

        text = item['text']
        predict = item['pred_label']
        confid = item['confidence']

        if actual is not None:
            
            if actual == predict:
                compare = 'Matched !!'
                correct += 1
            else:
                compare = 'Wrong !!'
                incorrect+= 1

        else:

            actual = '--'
            compare = ''

        
        print(f"{compare}\n{text}\nPredicted: {predict}\nActual: {actual}\n")

    return correct,incorrect
    

# Get Data

In [125]:
positive_texts = [
    "Company reported strong earnings and raised guidance.",
    "Apple beats Q3 estimates with record iPhone sales.",
    "Tesla shares surge after strong delivery numbers.",
    "Fed signals pause in rate hikes, markets rally.",
    "Amazon announces $10 billion stock buyback program.",
    "Nvidia profit triples on surging AI chip demand.",
    "Goldman Sachs upgrades stock to buy with $200 target.",
]

negative_texts = [
    "Revenue missed expectations and outlook remains weak.",
    "Company slashes full-year forecast amid supply chain issues.",
    "Bank reports massive write-down on bad loans.",
    "Layoffs announced as tech firm struggles with slowing growth.",
    "Regulators probe firm over accounting irregularities.",
    "Oil prices plunge on recession fears and weak demand.",
    "Credit rating downgraded to junk status by Moody's.",
]

neutral_texts = [
    "Stock traded flat after mixed quarterly results.",
    "Board meeting scheduled for next Tuesday.",
    "Company files annual report with the SEC.",
    "CEO to present at investor conference next month.",
    "Merger expected to close in Q4 pending regulatory approval.",
    "Analyst maintains hold rating with unchanged price target.",
    "Trading volume was below average in afternoon session.",
]


# Predict

In [126]:
correct_pos,incorrect_pos = main_sentiment_predict(positive_texts , 'positive')
correct_neg,incorrect_neg = main_sentiment_predict(negative_texts , 'negative')
correct_neu,incorrect_neu = main_sentiment_predict(neutral_texts , 'neutral')

Wrong !!
Company reported strong earnings and raised guidance.
Predicted: neutral
Actual: positive

Matched !!
Apple beats Q3 estimates with record iPhone sales.
Predicted: positive
Actual: positive

Wrong !!
Tesla shares surge after strong delivery numbers.
Predicted: neutral
Actual: positive

Wrong !!
Fed signals pause in rate hikes, markets rally.
Predicted: neutral
Actual: positive

Matched !!
Amazon announces $10 billion stock buyback program.
Predicted: positive
Actual: positive

Wrong !!
Nvidia profit triples on surging AI chip demand.
Predicted: neutral
Actual: positive

Wrong !!
Goldman Sachs upgrades stock to buy with $200 target.
Predicted: neutral
Actual: positive

Wrong !!
Company reported strong earnings and raised guidance.
Predicted: neutral
Actual: negative

Wrong !!
Apple beats Q3 estimates with record iPhone sales.
Predicted: neutral
Actual: negative

Wrong !!
Tesla shares surge after strong delivery numbers.
Predicted: neutral
Actual: negative

Wrong !!
Fed signals 

In [127]:
print(f'for Positive: Correct - {correct_pos} and Incorrect = {incorrect_pos}')
print(f'for Negative: Correct - {correct_neg} and Incorrect = {incorrect_neg}')
print(f'for Neutral: Correct - {correct_neu} and Incorrect = {incorrect_neu}')

for Positive: Correct - 2 and Incorrect = 5
for Negative: Correct - 0 and Incorrect = 7
for Neutral: Correct - 2 and Incorrect = 5
